# Efficient ResNet on CIFAR-10 — Colab runner

1. 把数据压缩包 `kaggle_data.zip`（`train_images/`、`test_images/`、`train_labels.csv`、`test.csv`、`sample_submission.csv`）上传到 Google Drive 的 `MyDrive/` 根目录。
2. 菜单 **Runtime → Change runtime type → GPU**（T4 即可）。
3. 从上到下依次运行。checkpoint 和 `submission.csv` 会保存到 Drive 的 `MyDrive/kaggle_DL_output/`，断线也不会丢。

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = 'https://github.com/yzeng018-cpu/DeepLearningProject1.git'
DATA_ZIP   = '/content/drive/MyDrive/kaggle_data.zip'         # 上传数据 zip 的位置
OUTPUT_DIR = '/content/drive/MyDrive/kaggle_DL_output'        # checkpoint / submission 保存位置
WORK_DIR   = '/content/project'                               # 本地磁盘，读图片比 Drive 快得多

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
!rm -rf {WORK_DIR} && git clone -q {REPO_URL} {WORK_DIR}
!unzip -q "{DATA_ZIP}" -d {WORK_DIR}
%cd {WORK_DIR}
!ls
!ls train_images | wc -l && ls test_images | wc -l

In [ ]:
# 检查参数量 (<= 5M)
!python model.py

In [ ]:
# 训练：90% train / 10% val，按验证准确率保存 best.pt
# 用全部 5 万张训练最终模型时加 --val-frac 0
!python train.py --epochs 100 --out "{OUTPUT_DIR}/checkpoints" 2>&1 | tee "{OUTPUT_DIR}/train_log.txt"

In [ ]:
# 生成提交文件（test_images 只用于推理）
!python predict.py --ckpt "{OUTPUT_DIR}/checkpoints/best.pt" --out "{OUTPUT_DIR}/submission.csv"
!head -5 "{OUTPUT_DIR}/submission.csv" && wc -l "{OUTPUT_DIR}/submission.csv"

In [ ]:
# 可选：下载到本地
from google.colab import files
files.download(f'{OUTPUT_DIR}/submission.csv')